In [12]:
import pandas as pd
from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

tqdm.pandas()

In [15]:
cb_model = CatBoostClassifier().load_model("weights/catboost_model")
embedding_model = SentenceTransformer("mchochlov/codebert-base-cd-ft")

In [16]:
df_hack = pd.read_csv("../data/dataset.csv", index_col=0)
df_hack.head()

,action_id,time,session_id,kernel_id,notebook_name,event,cell_index,cell_num,cell_type,cell_source,cell_output,user_id,expert
0,0,2023-05-06 10:32:26.282,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,save_notebook,NaN,NaN,NaN,"[\n {\n ""id"": ""35c0b3b694f84140846a21197ea...",NaN,student_7,False
1,1,2023-05-06 10:32:55.892,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False
2,2,2023-05-06 10:32:56.229,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,finished_execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,"[{""output_type"":""stream"",""size"":23}]",student_7,False
3,3,2023-05-06 10:32:58.048,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False
4,4,2023-05-06 10:32:58.429,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,finished_execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,"[{""output_type"":""stream"",""size"":23}]",student_7,False


In [17]:
embeddings = embedding_model.encode(df_hack.cell_source.fillna("").tolist(), show_progress_bar=True)
df_hack["cell_source_embedding"] = list(embeddings)

Batches:   0%|          | 0/739 [00:00<?, ?it/s]

In [18]:
df_hack["cell_source_embedding"].to_csv("data/processed/codebert_hackathon_cell_source_embeddings.csv")

In [19]:
predictions = cb_model.predict(df_hack.cell_source_embedding.tolist())
predictions = [p[0] for p in predictions]

In [22]:
df_hack["cell_label"] = predictions
df_hack[df_hack["event"] == "execute"].head()

,action_id,time,session_id,kernel_id,notebook_name,event,cell_index,cell_num,cell_type,cell_source,cell_output,user_id,expert,cell_source_embedding,cell_label
1,1,2023-05-06 10:32:55.892,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,"[-0.32941195, 0.11186513, 0.47475833, 0.078203...",helper_functions
3,3,2023-05-06 10:32:58.048,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,"[-0.32941195, 0.11186513, 0.47475833, 0.078203...",helper_functions
5,5,2023-05-06 10:33:01.263,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,"[-0.34601116, 0.09529341, 0.50744224, 0.088523...",helper_functions
6,6,2023-05-06 10:33:06.253,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,"[-0.32941148, 0.11186546, 0.4747587, 0.0782040...",helper_functions
8,8,2023-05-06 10:33:09.812,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,"[-0.32941148, 0.11186546, 0.4747587, 0.0782040...",helper_functions


In [23]:
df_hack[["action_id", "cell_label"]].to_csv("../data/labels_mapping_catboost.csv")
df_hack[["action_id", "cell_label"]].head()

,action_id,cell_label
0,0,data_preprocessing
1,1,helper_functions
2,2,helper_functions
3,3,helper_functions
4,4,helper_functions
